# BPL OCR · unified efficiency measurement

Use one model per **fresh hosted Colab runtime**. The existing OCR notebook, source scans and
results remain unchanged. This notebook runs **fresh inference**, including for models with
existing completed predictions. Existing results supply immutable model revisions only.

All five models receive identical original line crops, batch size 1, with one discarded warmup.
Default scope: 1,573 shared detections on pages 161–167; page 161 includes only its directory body.
Tesseract is measured as **line PSM 7**, with an engine process per line; historical page PSM 3
timings are a different condition. Device/precision settings follow the existing adapters.

## 1. Choose a model and set up Colab

Edit `ENGINE`, then run both setup/helper cells. Reset the runtime before switching models.
Tesseract/Paddle use CPU; dots/Qwen require GPU; EasyOCR uses available CPU/GPU.
Setup seconds include dependency installation. Download plus model loading is measured separately.

In [ ]:
import time
_METRIC_SETUP_START = time.perf_counter()
ENGINE = 'easyocr'  # tesseract, dots_mocr, easyocr, paddleocr, qwen3_vl_2b

import subprocess
import sys
from pathlib import Path

if not Path('/content').exists():
    raise RuntimeError('Use a hosted Google Colab runtime.')
if ENGINE not in ('tesseract', 'dots_mocr', 'easyocr', 'paddleocr', 'qwen3_vl_2b'):
    raise ValueError('Choose one of the five documented model names.')
if '_SETUP_ENGINE' in globals() and ENGINE != _SETUP_ENGINE:
    raise RuntimeError('Reset the runtime before changing ENGINE.')
dependencies = ['Pillow', 'numpy', 'psutil']
if ENGINE == 'tesseract':
    subprocess.run(['apt-get', 'update', '-qq'], check=True)
    subprocess.run(['apt-get', 'install', '-y', '-qq', 'tesseract-ocr'], check=True)
elif ENGINE == 'easyocr':
    dependencies += ['easyocr==1.7.2']
elif ENGINE == 'paddleocr':
    dependencies += ['paddlepaddle==3.2.0', 'paddleocr==3.7.0', 'huggingface-hub==0.36.2']
else:
    dependencies += ['transformers==4.57.6', 'accelerate==1.15.0', 'huggingface-hub==0.36.2']
    if ENGINE == 'dots_mocr':
        dependencies += ['qwen-vl-utils==0.0.14']
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--disable-pip-version-check', *dependencies], check=True)
DEVICE = 'cpu' if ENGINE in ('tesseract', 'paddleocr') else 'auto'
if ENGINE in ('dots_mocr', 'qwen3_vl_2b'):
    import torch
    if not torch.cuda.is_available():
        raise RuntimeError('Select Runtime > Change runtime type > GPU.')
_SETUP_ENGINE = ENGINE
print('Selected:', ENGINE, '| device:', DEVICE)
if ENGINE not in ('tesseract', 'paddleocr'):
    import torch
    DEVICE = 'cuda:0' if torch.cuda.is_available() else 'cpu'
SETUP_SECONDS = time.perf_counter() - _METRIC_SETUP_START
print('Measured setup seconds:', SETUP_SECONDS, '| actual device:', DEVICE)

In [ ]:
_METRIC_SOURCES = {'ocr_pipeline.py': '"""One BPL OCR pipeline: common scans/line geometry, five engines, line text only.\n\nHosted Colab runs recognition. Local CLI only builds the notebook or packages\nthe common data folder. No accuracy scoring, field extraction or gold labels.\n"""\nimport argparse\nimport csv\nimport hashlib\nimport io\nimport json\nfrom pathlib import Path\nimport shutil\nimport tempfile\nimport time\nimport zipfile\n\nFORMAT = "bpl-ocr-lines-v1"\nENGINES = ("tesseract", "dots_mocr", "easyocr", "paddleocr", "qwen3_vl_2b")\nTEXT_FIELDS = ("tesseract_text", "mocr_text", "easyocr_text", "paddleocr_text", "qwen_text")\nPAGES = list(range(161, 168))\n\n\ndef sha_file(path):\n    return hashlib.sha256(Path(path).read_bytes()).hexdigest()\n\n\ndef sha_text(text):\n    return hashlib.sha256(text.encode("utf-8")).hexdigest()\n\n\ndef identity(value):\n    return sha_text(json.dumps(value, sort_keys=True, ensure_ascii=False))\n\n\ndef write_json(path, value):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    path.write_text(json.dumps(value, indent=2, ensure_ascii=False) + "\\n", encoding="utf-8")\n\n\ndef source_identity(index):\n    return identity({"pages": index["pages"], "lines": index["lines"], "scope": index["scope"]})\n\n\ndef load_sources(root):\n    root = Path(root)\n    index = json.loads((root / "source/index.json").read_text(encoding="utf-8"))\n    if index.get("format") != FORMAT or [p["printed_page"] for p in index["pages"]] != PAGES:\n        raise ValueError("Expected the common printed-page 161–167 source folder")\n    if index.get("source_id") != source_identity(index):\n        raise ValueError("Source metadata or shared line geometry changed")\n    from PIL import Image\n    pages = {p["printed_page"]: p for p in index["pages"]}\n    for p in index["pages"]:\n        if p["image"] != f"source/p{p[\'printed_page\']}.jpg":\n            raise ValueError("Unexpected source image path")\n        image_path = root / p["image"]\n        if sha_file(image_path) != p["sha256"]:\n            raise ValueError("Source image changed: " + p["image"])\n        with Image.open(image_path) as image:\n            if list(image.size) != [p["width"], p["height"]]:\n                raise ValueError("Source dimensions changed")\n            image.verify()\n    seen = set()\n    for row in index["lines"]:\n        page = pages[row["printed_page"]]\n        x0, y0, x1, y1 = row["bbox"]\n        if (row["line_id"] in seen or not (0 <= x0 < x1 <= page["width"] and 0 <= y0 < y1 <= page["height"])):\n            raise ValueError("Duplicate line ID or invalid source coordinates")\n        seen.add(row["line_id"])\n    return index\n\n\ndef visible_lines(index):\n    """Keep existing order; page 161 excludes the abbreviation key and footer."""\n    return [r for r in index["lines"] if r["printed_page"] != 161 or 1645 <= r["bbox"][1] < 2678]\n\n\ndef line_identity(index, row):\n    page = next(p for p in index["pages"] if p["printed_page"] == row["printed_page"])\n    return identity({"source_sha256": page["sha256"], "line_id": row["line_id"], "bbox": row["bbox"]})\n\n\ndef read_jsonl(path):\n    result = []\n    if Path(path).exists():\n        for line in Path(path).read_text(encoding="utf-8").splitlines():\n            try:\n                value = json.loads(line)\n                if isinstance(value, dict):\n                    result.append(value)\n            except ValueError:\n                pass  # Retain usable records when an interrupted append is incomplete.\n    return result\n\n\ndef read_predictions(root, index, engine):\n    if engine not in ENGINES:\n        raise ValueError("Unknown engine")\n    expected = {r["line_id"]: r for r in index["lines"]}\n    models_path = Path(root) / "results/models.json"\n    models = json.loads(models_path.read_text()) if models_path.exists() else {}\n    metadata = models.get(engine, {})\n    result = {}\n    for record in read_jsonl(Path(root) / "results/raw" / (engine + ".jsonl")):\n        row = expected.get(record.get("line_id"))\n        text = record.get("raw_text")\n        if row is None or not isinstance(text, str) or sha_text(text) != record.get("raw_text_sha256"):\n            continue\n        if record.get("input_id") is not None:\n            matches = record["input_id"] == line_identity(index, row)\n        else:\n            # Native v4 mOCR log is preserved byte-for-byte during consolidation.\n            matches = (metadata.get("imported_source_id") == index["source_id"]\n                       and record.get("bbox") == row["bbox"])\n        if matches and record.get("status") in ("completed", "empty", "failed", "truncated", "out_of_memory", "detected_no_text"):\n            result[row["line_id"]] = record\n    return result\n\n\ndef export_lines(root, index=None):\n    root = Path(root)\n    index = load_sources(root) if index is None else index\n    predictions = {e: read_predictions(root, index, e) for e in ENGINES}\n    pages = {p["printed_page"]: p for p in index["pages"]}\n    path = root / "results/lines.csv"\n    path.parent.mkdir(parents=True, exist_ok=True)\n    counts = {}\n    with path.open("w", encoding="utf-8-sig", newline="") as handle:\n        writer = csv.DictWriter(handle, fieldnames=[*TEXT_FIELDS, "sources"])\n        writer.writeheader()\n        for row in visible_lines(index):\n            out = {field: " ".join(predictions[e].get(row["line_id"], {}).get("raw_text", "").split())\n                   for e, field in zip(ENGINES, TEXT_FIELDS)}\n            out["sources"] = pages[row["printed_page"]]["source_page_url"]\n            writer.writerow(out)\n    for engine in ENGINES:\n        counts[engine] = {status: sum(predictions[engine].get(r["line_id"], {}).get("status", "not_run") == status\n                                     for r in visible_lines(index))\n                          for status in ("completed", "empty", "failed", "truncated", "out_of_memory", "detected_no_text", "not_run")}\n    write_json(root / "results/status.json", {"format": FORMAT, "source_id": index["source_id"],\n               "rows": len(visible_lines(index)), "models": counts,\n               "meaning": "One shared detected scan line per row, not one person. Blank text: inspect raw/status records. No accuracy measures."})\n    return path\n\n\ndef import_bundle(archive, root):\n    root = Path(root)\n    if root.exists() and any(root.iterdir()):\n        raise FileExistsError("Output folder is populated. Continue with the loaded ROOT, or use a fresh Colab runtime.")\n    with tempfile.TemporaryDirectory() as temporary:\n        staging = Path(temporary) / "data"\n        with zipfile.ZipFile(archive) as z:\n            files = [i for i in z.infolist() if not i.is_dir() and not any(p.startswith(".") or p == "__MACOSX" for p in Path(i.filename).parts)]\n            indexes = [i.filename for i in files if i.filename.endswith("source/index.json")]\n            if len(indexes) != 1 or sum(i.file_size for i in files) > 500_000_000:\n                raise ValueError("Upload one common BPL OCR folder ZIP (source/index.json required)")\n            prefix = indexes[0][:-len("source/index.json")]\n            for info in files:\n                if not info.filename.startswith(prefix):\n                    continue\n                relative = Path(info.filename[len(prefix):])\n                if relative.is_absolute() or ".." in relative.parts:\n                    raise ValueError("Unsafe ZIP path")\n                if (not relative.parts or relative.parts[0] not in ("source", "results")\n                        or relative.suffix not in (".jpg", ".json", ".jsonl", ".csv", ".tsv", ".txt")):\n                    continue\n                if info.file_size > 100_000_000:\n                    raise ValueError("ZIP member too large")\n                target = staging / relative\n                target.parent.mkdir(parents=True, exist_ok=True)\n                target.write_bytes(z.read(info))\n        index = load_sources(staging)\n        shutil.copytree(staging, root, dirs_exist_ok=True)\n    export_lines(root, index)\n    return index\n\n\ndef pack_bundle(root, destination):\n    """Create an on-demand transfer ZIP; keep source images once in the workspace."""\n    root, destination = Path(root), Path(destination)\n    if root.resolve() in destination.resolve().parents:\n        raise ValueError("Write the transfer ZIP outside the data folder")\n    destination.parent.mkdir(parents=True, exist_ok=True)\n    temporary = destination.with_suffix(destination.suffix + ".part")\n    with zipfile.ZipFile(temporary, "w", zipfile.ZIP_DEFLATED) as archive:\n        for directory in ("source", "results"):\n            for path in sorted((root / directory).rglob("*")):\n                if path.is_file() and not any(p.startswith(".") for p in path.relative_to(root).parts):\n                    archive.write(path, path.relative_to(root))\n    temporary.replace(destination)\n    return destination\n\n\ndef run_ocr(root, engine, *, device="auto", max_new_lines=None, force=False, adapter_factory=None):\n    """Append exact per-line outputs and checkpoint the same CSV for every model."""\n    root = Path(root)\n    if engine not in ENGINES:\n        raise ValueError("Choose one of: " + ", ".join(ENGINES))\n    if max_new_lines is not None and (type(max_new_lines) is not int or max_new_lines <= 0):\n        raise ValueError("max_new_lines must be None or a positive integer")\n    index = load_sources(root)\n    lines = visible_lines(index)\n    old = read_predictions(root, index, engine)\n    pending = [r for r in lines if force or old.get(r["line_id"], {}).get("status") != "completed"]\n    if max_new_lines is not None:\n        pending = pending[:max_new_lines]\n    print(engine, ":", len(lines), "source lines;", len(pending), "to recognize this run.", flush=True)\n    if not pending:\n        export_lines(root, index)\n        return {"engine": engine, "new_lines": 0, "message": "Existing completed outputs retained; no model loaded."}\n    models_path = root / "results/models.json"\n    models = json.loads(models_path.read_text()) if models_path.exists() else {}\n    prior = models.get(engine, {})\n    pinned = prior.get("settings_history", {}).get(prior.get("active_settings_id"), {})\n    adapter = (adapter_factory or build_recognizer)(engine, device=device, pinned_settings=pinned,\n                                                   model_cache=root.parent / "bpl_ocr_model_cache")\n    settings = {**adapter.settings, "pipeline_code_sha256": sha_file(__file__)}\n    settings_id = identity(settings)\n    history = dict(prior.get("settings_history", {}))\n    history[settings_id] = settings\n    models[engine] = {**prior, "active_settings_id": settings_id, "settings_history": history,\n                      "last_runtime": adapter.metadata,\n                      "input": "Shared original scan line crops. Existing imports retain their historical context in settings_history."}\n    write_json(models_path, models)\n    pages = {p["printed_page"]: p for p in index["pages"]}\n    log_path = root / "results/raw" / (engine + ".jsonl")\n    log_path.parent.mkdir(parents=True, exist_ok=True)\n    active_page, image, attempted = None, None, 0\n    from PIL import Image\n    try:\n        with log_path.open("a", encoding="utf-8") as handle:\n            if log_path.stat().st_size and not log_path.read_bytes().endswith(b"\\n"):\n                handle.write("\\n")\n            for row in pending:\n                if active_page != row["printed_page"]:\n                    if image is not None:\n                        image.close()\n                    page = pages[row["printed_page"]]\n                    if sha_file(root / page["image"]) != page["sha256"]:\n                        raise ValueError("Source image changed during OCR")\n                    with Image.open(root / page["image"]) as source:\n                        image = source.convert("RGB")\n                    active_page = row["printed_page"]\n                record = {"line_id": row["line_id"], "bbox": row["bbox"], "input_id": line_identity(index, row),\n                          "settings_id": settings_id}\n                try:\n                    with image.crop(tuple(row["bbox"])) as tile:\n                        result = adapter.predict(tile)\n                    if not isinstance(result.get("raw_text"), str) or result.get("status") not in ("completed", "empty", "truncated"):\n                        raise ValueError("Unexpected recognition response")\n                    record.update(result)\n                except Exception as error:\n                    oom = "out of memory" in str(error).lower() or "outofmemory" in type(error).__name__.lower()\n                    record.update(raw_text="", status="out_of_memory" if oom else "failed",\n                                  error=type(error).__name__ + ": " + str(error))\n                record["raw_text_sha256"] = sha_text(record["raw_text"])\n                handle.write(json.dumps(record, ensure_ascii=False) + "\\n")\n                handle.flush()\n                attempted += 1\n                print(engine, row["line_id"], record["status"], f"({attempted}/{len(pending)})", flush=True)\n                if attempted % 20 == 0:\n                    export_lines(root, index)\n                if record["status"] == "out_of_memory":\n                    break\n    except KeyboardInterrupt:\n        print("Interrupted. Completed lines are saved. Run the download cell before leaving Colab.")\n    finally:\n        if image is not None:\n            image.close()\n        try:\n            adapter.close()\n        finally:\n            export_lines(root, index)\n    return {"engine": engine, "new_lines": attempted, "csv": str(root / "results/lines.csv")}\n\n\n"""Recognition adapters for the common BPL line-image pipeline.\n\nOnly standard-library imports execute at import time. OCR frameworks and model\nweights are loaded only by build_recognizer(), in the hosted Colab runtime.\nEvery recognizer sees the same original source crop, never another OCR\'s text.\n"""\nimport base64\nimport gc\nimport hashlib\nimport importlib.metadata\nimport io\nfrom pathlib import Path\nimport platform\nimport re\nimport shutil\nimport subprocess\nimport urllib.request\n\nRECOGNIZER_KEYS = ("tesseract", "dots_mocr", "easyocr", "paddleocr", "qwen3_vl_2b")\nTESSDATA_REVISION = "e12c65a915945e4c28e237a9b52bc4a8f39a0cec"\nTESSDATA_SHA256 = "8280aed0782fe27257a68ea10fe7ef324ca0f8d85bd2fd145d1c2b560bcb66ba"\nDOTS_REVISION = "e539fbb52280393adc081b289ec597430a0f9031"\nDOTS_PROMPT = "Extract the text content from this image."\n\n\ndef _model_packages(*names):\n    result = {}\n    for name in names:\n        try:\n            result[name] = importlib.metadata.version(name)\n        except importlib.metadata.PackageNotFoundError:\n            result[name] = None\n    return result\n\n\ndef _model_file_sha(path):\n    digest = hashlib.sha256()\n    with Path(path).open("rb") as handle:\n        for chunk in iter(lambda: handle.read(1024 * 1024), b""):\n            digest.update(chunk)\n    return digest.hexdigest()\n\n\ndef _model_revision(hub, repo, stored=None):\n    revision = stored if stored is not None else hub.model_info(repo).sha\n    if not isinstance(revision, str) or not re.fullmatch(r"[0-9a-f]{40}", revision):\n        raise ValueError("Missing or invalid immutable model revision: " + repo)\n    return revision\n\n\ndef _recognized(raw_text, **extra):\n    return {"raw_text": raw_text, "status": "completed" if raw_text.strip() else "empty", **extra}\n\n\ndef _generation_result(raw, tokens, eos, limit):\n    eos = eos if isinstance(eos, list) else [eos]\n    truncated = len(tokens) >= limit and (not tokens or tokens[-1] not in eos)\n    return {"raw_text": raw,\n            "status": "truncated" if truncated else ("completed" if raw.strip() else "empty"),\n            "extra": {"generated_tokens": len(tokens), "generated_token_ids": tokens}}\n\n\nclass _LineRecognizer:\n    def __init__(self, engine, device, framework=None, packages=None):\n        self.engine, self.device, self.framework = engine, device, framework\n        self.gpu = device.startswith(("gpu", "cuda"))\n        self.settings = {"engine": engine, "adapter_version": 1, "input": "one original RGB source-line crop",\n                         "device_type": "cuda" if self.gpu else "cpu", "precision": "float32",\n                         "packages": packages or {}}\n        self.metadata = {"actual_device": device, "platform": platform.platform(),\n                         "processor": platform.processor(), "packages": packages or {}}\n        self.model = None\n\n    def close(self):\n        if self.engine == "paddleocr" and hasattr(self.model, "close"):\n            self.model.close()\n        self.model = None\n        if hasattr(self, "processor"):\n            self.processor = None\n        gc.collect()\n        if self.gpu:\n            cuda = self.framework.device.cuda if self.engine == "paddleocr" else self.framework.cuda\n            if hasattr(cuda, "empty_cache"):\n                cuda.empty_cache()\n\n\nclass _TesseractRecognizer(_LineRecognizer):\n    def __init__(self, model_cache):\n        super().__init__("tesseract", "cpu", packages=_model_packages("Pillow"))\n        self.binary = shutil.which("tesseract")\n        if not self.binary:\n            raise RuntimeError("Run the Colab Tesseract setup cell first")\n        self.data_dir = Path(model_cache) / "tessdata_best" / TESSDATA_REVISION\n        self.data_dir.mkdir(parents=True, exist_ok=True)\n        target = self.data_dir / "eng.traineddata"\n        if not target.exists() or _model_file_sha(target) != TESSDATA_SHA256:\n            url = f"https://raw.githubusercontent.com/tesseract-ocr/tessdata_best/{TESSDATA_REVISION}/eng.traineddata"\n            request = urllib.request.Request(url, headers={"User-Agent": "BPL-OCR-Pipeline/1.0"})\n            partial = target.with_suffix(".part")\n            try:\n                with urllib.request.urlopen(request, timeout=60) as response, partial.open("wb") as handle:\n                    shutil.copyfileobj(response, handle)\n                if _model_file_sha(partial) != TESSDATA_SHA256:\n                    raise ValueError("Pinned English Tesseract weights checksum mismatch")\n                partial.replace(target)\n            finally:\n                partial.unlink(missing_ok=True)\n        version = subprocess.run([self.binary, "--version"], capture_output=True, text=True, check=True).stdout.splitlines()[0]\n        self.settings.update({"tesseract_version": version, "language": "eng", "oem": 1, "psm": 7,\n                              "english_revision": TESSDATA_REVISION, "english_sha256": TESSDATA_SHA256,\n                              "preprocessing": {"mode": "L", "scale": 2, "resampling": "LANCZOS", "white_border": 10}})\n\n    def predict(self, image):\n        from PIL import Image, ImageOps\n        crop = image.convert("L")\n        crop = crop.resize((crop.width*2, crop.height*2), Image.Resampling.LANCZOS)\n        crop = ImageOps.expand(crop, border=10, fill=255)\n        encoded = io.BytesIO()\n        crop.save(encoded, format="PNG")\n        result = subprocess.run([self.binary, "stdin", "stdout", "--tessdata-dir", str(self.data_dir),\n                                 "-l", "eng", "--oem", "1", "--psm", "7"],\n                                input=encoded.getvalue(), capture_output=True, check=True)\n        return _recognized(result.stdout.decode("utf-8"), extra={"stderr": result.stderr.decode("utf-8", errors="replace")})\n\n\nclass _EasyRecognizer(_LineRecognizer):\n    def __init__(self, device, torch, np, easyocr, model_cache):\n        super().__init__("easyocr", device, torch, _model_packages("easyocr", "torch", "torchvision", "numpy", "Pillow"))\n        self.np = np\n        self.model = easyocr.Reader(["en"], gpu=device if self.gpu else False,\n                                    detector=False, recog_network="english_g2", quantize=False,\n                                    cudnn_benchmark=False, model_storage_directory=str(Path(model_cache) / "easyocr"))\n        model_file = Path(self.model.model_storage_directory) / "english_g2.pth"\n        self.settings.update({"recognizer": "english_g2", "languages": ["en"], "detector": False,\n            "quantize": False, "cudnn_benchmark": False, "model_sha256": _model_file_sha(model_file),\n            "recognition": {"decoder": "greedy", "batch_size": 1, "workers": 0, "detail": 1,\n                            "paragraph": False, "contrast_ths": 0.1, "adjust_contrast": 0.5, "filter_ths": 0.003}})\n        self.metadata["actual_device"] = str(self.model.device)\n        if self.gpu:\n            self.metadata["gpu_name"] = torch.cuda.get_device_name()\n\n    def predict(self, image):\n        predictions = self.model.recognize(self.np.asarray(image.convert("L")), **self.settings["recognition"])\n        records = [{"polygon": [[float(x), float(y)] for x, y in box],\n                    "text": str(text), "confidence": float(confidence)} for box, text, confidence in predictions]\n        return _recognized("\\n".join(r["text"] for r in records), extra={"recognition_results": records})\n\n\nclass _PaddleRecognizer(_LineRecognizer):\n    def __init__(self, device, paddle, np, paddleocr, hub, pinned_settings=None):\n        super().__init__("paddleocr", device, paddle,\n                         _model_packages("paddleocr", "paddlex", "paddlepaddle", "paddlepaddle-gpu", "numpy", "Pillow"))\n        self.np = np\n        paddle.set_device(device)\n        repo, name = "PaddlePaddle/en_PP-OCRv5_mobile_rec", "en_PP-OCRv5_mobile_rec"\n        revision = _model_revision(hub, repo, (pinned_settings or {}).get("revision"))\n        folder = Path(hub.snapshot_download(repo, revision=revision,\n                      allow_patterns=["config.json", "inference.json", "inference.yml", "inference.pdiparams"]))\n        hashes = {p.name: _model_file_sha(p) for p in sorted(folder.iterdir()) if p.is_file()}\n        backend = {"enable_hpi": False, "engine": "paddle_static", "use_tensorrt": False,\n                   "precision": "fp32", "enable_mkldnn": False, "cpu_threads": 2, "enable_cinn": False}\n        self.model = paddleocr.TextRecognition(model_name=name, model_dir=str(folder), device=device, **backend)\n        self.settings.update({"model": repo, "recognizer": name, "revision": revision,\n                              "model_sha256": hashes, "input_channels": "BGR", "batch_size": 1, "backend": backend})\n        self.metadata["actual_device"] = str(paddle.get_device())\n        if self.gpu:\n            self.metadata["gpu_name"] = paddle.device.cuda.get_device_name()\n\n    def predict(self, image):\n        bgr = self.np.asarray(image.convert("RGB"))[:, :, ::-1].copy()\n        results = list(self.model.predict(input=bgr, batch_size=1))\n        if len(results) != 1:\n            raise ValueError("Expected one Paddle recognition result for one source crop")\n        return _recognized(str(results[0]["rec_text"]), extra={"confidence": float(results[0]["rec_score"])})\n\n\nclass _QwenRecognizer(_LineRecognizer):\n    def __init__(self, device, torch, transformers, hub, pinned_settings=None):\n        super().__init__("qwen3_vl_2b", device, torch,\n                         _model_packages("transformers", "torch", "torchvision", "accelerate", "numpy", "Pillow"))\n        repo = "Qwen/Qwen3-VL-2B-Instruct"\n        revision = _model_revision(hub, repo, (pinned_settings or {}).get("revision"))\n        dtype = torch.float16 if self.gpu else torch.float32\n        self.model = transformers.Qwen3VLForConditionalGeneration.from_pretrained(\n            repo, revision=revision, dtype=dtype, attn_implementation="sdpa", device_map=device).eval()\n        self.processor = transformers.AutoProcessor.from_pretrained(repo, revision=revision,\n                             use_fast=False, min_pixels=4096, max_pixels=262144)\n        prompt = ("Transcribe all text in this image faithfully. Preserve printed ditto marks, "\n                  "abbreviations, punctuation and line breaks. Do not expand, correct, infer, "\n                  "or explain anything. Output only the transcription.")\n        self.settings.update({"model": repo, "revision": revision, "dtype": str(dtype),\n            "precision": "float16" if self.gpu else "float32", "attention": "sdpa", "prompt": prompt,\n            "min_pixels": 4096, "max_pixels": 262144,\n            "generation": {"max_new_tokens": 256, "do_sample": False, "num_beams": 1, "repetition_penalty": 1.0},\n            "processor_sha256": hashlib.sha256(self.processor.image_processor.to_json_string().encode()).hexdigest(),\n            "generation_config": self.model.generation_config.to_dict()})\n        if self.gpu:\n            self.metadata["gpu_name"] = torch.cuda.get_device_name()\n\n    def predict(self, image):\n        messages = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": self.settings["prompt"]}]}]\n        text = self.processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)\n        inputs = self.processor(text=[text], images=[image.convert("RGB")], return_tensors="pt").to(self.model.device)\n        inputs.pop("token_type_ids", None)\n        with self.framework.inference_mode():\n            generated = self.model.generate(**inputs, **self.settings["generation"])\n        tokens = generated[0, inputs["input_ids"].shape[-1]:].tolist()\n        raw = self.processor.decode(tokens, skip_special_tokens=True, clean_up_tokenization_spaces=False)\n        return _generation_result(raw, tokens, self.model.generation_config.eos_token_id, 256)\n\n\nclass _DotsRecognizer(_LineRecognizer):\n    def __init__(self, device, torch, transformers, hub, model_cache, pinned_settings=None):\n        if not device.startswith("cuda"):\n            raise RuntimeError("dots.mocr requires a Colab GPU runtime")\n        super().__init__("dots_mocr", device, torch,\n                         _model_packages("torch", "transformers", "qwen-vl-utils", "accelerate", "huggingface-hub", "Pillow"))\n        repo = "dots-studio/dots.mocr"\n        revision = _model_revision(hub, repo, (pinned_settings or {}).get("revision", DOTS_REVISION))\n        local = Path(model_cache) / "dots_mocr" / revision\n        hub.snapshot_download(repo, revision=revision, local_dir=str(local))\n        hub.hf_hub_download(repo, "modeling_dots_vision.py", revision=revision, local_dir=str(local), force_download=True)\n        vision_path = local / "modeling_dots_vision.py"\n        source = vision_path.read_text()\n        original_sha = hashlib.sha256(source.encode()).hexdigest()\n        if \'"sdpa"\' not in source and "\'sdpa\'" not in source:\n            raise RuntimeError("The pinned dots vision code lacks the expected SDPA implementation")\n        guarded = "try:\\n    from flash_attn import flash_attn_varlen_func\\nexcept ImportError:\\n    flash_attn_varlen_func = None\\n"\n        if "from flash_attn import flash_attn_varlen_func" in source.splitlines():\n            vision_path.write_text(source.replace("from flash_attn import flash_attn_varlen_func", guarded, 1))\n        config = transformers.AutoConfig.from_pretrained(str(local), trust_remote_code=True)\n        if isinstance(config.vision_config, dict):\n            config.vision_config["attn_implementation"] = "sdpa"\n        else:\n            config.vision_config.attn_implementation = "sdpa"\n        self.model = transformers.AutoModelForCausalLM.from_pretrained(str(local), config=config,\n            trust_remote_code=True, torch_dtype=torch.bfloat16, attn_implementation="sdpa",\n            device_map={"": device}, low_cpu_mem_usage=True).eval()\n        self.processor = transformers.AutoProcessor.from_pretrained(str(local), trust_remote_code=True)\n        self.settings.update({"model": repo, "revision": revision, "prompt": DOTS_PROMPT,\n            "dtype": "torch.bfloat16", "precision": "bfloat16", "attention": "sdpa", "max_pixels": 262144,\n            "generation": {"max_new_tokens": 256, "do_sample": False},\n            "original_vision_code_sha256": original_sha, "used_vision_code_sha256": _model_file_sha(vision_path)})\n        self.metadata["gpu_name"] = torch.cuda.get_device_name()\n        self.metadata["native_bfloat16_supported"] = bool(torch.cuda.is_bf16_supported())\n\n    def predict(self, image):\n        from qwen_vl_utils import process_vision_info\n        encoded = io.BytesIO()\n        image.save(encoded, format="PNG")\n        data_url = "data:image/png;base64," + base64.b64encode(encoded.getvalue()).decode()\n        messages = [{"role": "user", "content": [\n            {"type": "image", "image": data_url, "max_pixels": 262144},\n            {"type": "text", "text": DOTS_PROMPT}]}]\n        chat = self.processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)\n        images, videos = process_vision_info(messages)\n        inputs = self.processor(text=[chat], images=images, videos=videos, padding=True, return_tensors="pt").to(self.device)\n        with self.framework.inference_mode():\n            output = self.model.generate(**inputs, **self.settings["generation"])\n        tokens = output[0, inputs.input_ids.shape[1]:].tolist()\n        raw = self.processor.decode(tokens, skip_special_tokens=True, clean_up_tokenization_spaces=False)\n        result = _generation_result(raw, tokens, self.model.generation_config.eos_token_id, 256)\n        result["extra"]["processor_sizes"] = [list(im.size) for im in images]\n        return result\n\n\ndef build_recognizer(engine, device="auto", pinned_settings=None, model_cache=None):\n    """Load one recognition engine in hosted Colab; preserve immutable resume pins."""\n    if engine not in RECOGNIZER_KEYS:\n        raise ValueError("Unknown OCR engine: " + str(engine))\n    if device not in {"auto", "cpu", "gpu", "gpu:0", "cuda", "cuda:0"}:\n        raise ValueError("device must be auto, cpu or cuda:0")\n    model_cache = Path(model_cache or "/content/bpl_model_cache")\n    model_cache.mkdir(parents=True, exist_ok=True)\n    if engine == "tesseract":\n        return _TesseractRecognizer(model_cache)\n    if engine == "paddleocr":\n        import paddle\n        import paddleocr\n        import numpy as np\n        import huggingface_hub as hub\n        if device == "auto":\n            device = "gpu:0" if paddle.is_compiled_with_cuda() and paddle.device.cuda.device_count() else "cpu"\n        return _PaddleRecognizer("cpu" if device == "cpu" else "gpu:0", paddle, np, paddleocr, hub, pinned_settings)\n    import torch\n    if device == "auto":\n        device = "cuda:0" if torch.cuda.is_available() else "cpu"\n    device = "cpu" if device == "cpu" else "cuda:0"\n    if device != "cpu" and not torch.cuda.is_available():\n        raise RuntimeError("A CUDA GPU was requested but is not available")\n    if engine == "easyocr":\n        import easyocr\n        import numpy as np\n        return _EasyRecognizer(device, torch, np, easyocr, model_cache)\n    import transformers\n    import huggingface_hub as hub\n    if engine == "dots_mocr":\n        return _DotsRecognizer(device, torch, transformers, hub, model_cache, pinned_settings)\n    return _QwenRecognizer(device, torch, transformers, hub, pinned_settings)\n\n\n\ndef build_notebook():\n    """Embed this exact shared file into the sole hosted Colab OCR notebook."""\n    import nbformat as nbf\n    source = Path(__file__).read_text(encoding="utf-8")\n    cells = []\n    def md(text):\n        cells.append(nbf.v4.new_markdown_cell(text.strip()))\n    def code(text, name):\n        cell = nbf.v4.new_code_cell(text.strip())\n        cell["id"] = "ocr-" + name\n        cells.append(cell)\n    md("""# BPL OCR · pages 161–167\n\nOne pipeline, the same seven scans and shared detected line coordinates, five selectable models.\nRun one model in each **fresh hosted Colab runtime** and upload the latest data ZIP to accumulate outputs.\nNo local OCR installation or API key is required. The scope is OCR text only.\n\nThe supplied folder already includes the existing Tesseract and dots.mocr outputs.\nOther model columns remain empty until run. Historical Tesseract text used full-page context;\nfresh recognition in this notebook uses the same line crops as the other models.\nOne detected line is not necessarily one physical printed line or one person.\n""")\n    md("""## 1. Choose a model and set up Colab\n\nSet `ENGINE` before running. Use GPU for dots.mocr and Qwen; EasyOCR uses an available GPU.\nTesseract and this PaddleOCR setup use CPU. Reset the runtime before changing the model.\n""")\n    code("""ENGINE = \'easyocr\'  # tesseract, dots_mocr, easyocr, paddleocr, qwen3_vl_2b\n\nimport subprocess\nimport sys\nfrom pathlib import Path\n\nif not Path(\'/content\').exists():\n    raise RuntimeError(\'Use a hosted Google Colab runtime.\')\nif ENGINE not in (\'tesseract\', \'dots_mocr\', \'easyocr\', \'paddleocr\', \'qwen3_vl_2b\'):\n    raise ValueError(\'Choose one of the five documented model names.\')\nif \'_SETUP_ENGINE\' in globals() and ENGINE != _SETUP_ENGINE:\n    raise RuntimeError(\'Reset the runtime before changing ENGINE.\')\ndependencies = [\'Pillow\', \'numpy\']\nif ENGINE == \'tesseract\':\n    subprocess.run([\'apt-get\', \'update\', \'-qq\'], check=True)\n    subprocess.run([\'apt-get\', \'install\', \'-y\', \'-qq\', \'tesseract-ocr\'], check=True)\nelif ENGINE == \'easyocr\':\n    dependencies += [\'easyocr==1.7.2\']\nelif ENGINE == \'paddleocr\':\n    dependencies += [\'paddlepaddle==3.2.0\', \'paddleocr==3.7.0\', \'huggingface-hub==0.36.2\']\nelse:\n    dependencies += [\'transformers==4.57.6\', \'accelerate==1.15.0\', \'huggingface-hub==0.36.2\']\n    if ENGINE == \'dots_mocr\':\n        dependencies += [\'qwen-vl-utils==0.0.14\']\nsubprocess.run([sys.executable, \'-m\', \'pip\', \'install\', \'-q\', \'--disable-pip-version-check\', *dependencies], check=True)\nDEVICE = \'cpu\' if ENGINE in (\'tesseract\', \'paddleocr\') else \'auto\'\nif ENGINE in (\'dots_mocr\', \'qwen3_vl_2b\'):\n    import torch\n    if not torch.cuda.is_available():\n        raise RuntimeError(\'Select Runtime > Change runtime type > GPU.\')\n_SETUP_ENGINE = ENGINE\nprint(\'Selected:\', ENGINE, \'| device:\', DEVICE)\n""", "setup")\n    code("# Exact shared ocr_pipeline.py; no repository checkout is needed.\\n"\n         + "_PIPELINE_SOURCE = " + repr(source) + "\\n"\n         + "_CODE = Path(\'/content/bpl_ocr_code\')\\n_CODE.mkdir(exist_ok=True)\\n"\n         + "(_CODE / \'ocr_pipeline.py\').write_text(_PIPELINE_SOURCE, encoding=\'utf-8\')\\n"\n         + "sys.path.insert(0, str(_CODE))\\n"\n         + "from ocr_pipeline import import_bundle, run_ocr, export_lines, pack_bundle\\n", "helpers")\n    md("""## 2. Upload the common source/results ZIP\n\nUpload a ZIP of the common `fa26/data/ocr/` folder, or the latest ZIP downloaded below.\nIt contains `source/` and `results/`. Seven scans are kept once; line crops exist in memory only.\nThe existing coordinates are shared by all models, including any detection mistakes.\nPage 161 uses its lower directory body; pages 162–167 retain their existing detected lines.\n""")\n    code("""from google.colab import files\nuploaded = files.upload()\narchives = [name for name in uploaded if name.lower().endswith(\'.zip\')]\nif len(archives) != 1:\n    raise ValueError(\'Upload exactly one common source/results ZIP.\')\nuploaded_zip = Path(\'/content/bpl_uploaded.zip\')\nuploaded_zip.write_bytes(uploaded[archives[0]])\nROOT = Path(\'/content/bpl_ocr\')\nindex = import_bundle(uploaded_zip, ROOT)\nprint(\'Loaded seven pages. CSV:\', ROOT / \'results/lines.csv\')\n""", "inputs")\n    md("""## 3. Recognize lines\n\n`MAX_NEW_LINES = None` processes all remaining target lines; use `6` for a quick smoke test.\nCompleted outputs are retained by default, so existing Tesseract/mOCR results need no rerun.\nEmpty/failed/truncated outputs are retried. `FORCE = True` re-recognizes target lines;\nearlier raw responses and their original settings remain in the log.\n\nEvery model receives only the original line image. No other OCR text, name list or corrections\nenter recognition. Printed ditto marks and abbreviations are kept as recognized.\nInterrupt the cell if necessary, then run section 4 to download a checkpoint.\n""")\n    code("""MAX_NEW_LINES = None  # Set 6 for a short trial.\nFORCE = False\nif ENGINE != _SETUP_ENGINE:\n    raise RuntimeError(\'Reset the runtime before changing ENGINE.\')\nsummary = run_ocr(ROOT, ENGINE, device=DEVICE, max_new_lines=MAX_NEW_LINES, force=FORCE)\nprint(summary)\n""", "run")\n    md("""## 4. Download the same output folder\n\n`results/lines.csv` contains only the five model text columns and `sources`.\nBlank cells mean no available output; internal status/raw records distinguish unrun and failed cases.\nIt preserves one row per shared detected line, including failures. It does not join people or parse fields.\n\nThe ZIP also retains original scans, shared coordinates and raw outputs for resuming.\nUpload this latest ZIP in a fresh runtime for the next model. Download before Colab discards the VM.\n""")\n    code("""csv_path = export_lines(ROOT)\narchive = pack_bundle(ROOT, Path(\'/content/bpl_ocr.zip\'))\nprint(\'Line output:\', csv_path)\nfiles.download(str(archive))\n""", "export")\n    notebook = nbf.v4.new_notebook(cells=cells, metadata={\n        "kernelspec": {"name": "python3", "display_name": "Python 3", "language": "python"},\n        "language_info": {"name": "python"}, "accelerator": "GPU",\n        "colab": {"name": "ocr_colab.ipynb", "gpuType": "T4", "provenance": []}})\n    nbf.validate(notebook)\n    target = Path(__file__).with_name("ocr_colab.ipynb")\n    nbf.write(notebook, target)\n    return target\n\n\nif __name__ == "__main__":\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--notebook", action="store_true", help="Regenerate the single Colab notebook")\n    parser.add_argument("--pack", type=Path, help="Common source/results folder to package")\n    parser.add_argument("--output", type=Path, default=Path(tempfile.gettempdir()) / "bpl_ocr.zip")\n    args = parser.parse_args()\n    if args.notebook:\n        print(build_notebook())\n    elif args.pack:\n        print(pack_bundle(args.pack, args.output))\n    else:\n        parser.print_help()\n', 'ocr_efficiency.py': '"""Additive resource measurement for the existing BPL line OCR pipeline.\n\nOriginal source/, results/, adapters and recognition notebook are read-only.\nFresh inference and resource telemetry are stored only under efficiency/.\n"""\nimport argparse\nfrom collections import Counter\nfrom datetime import datetime, timezone\nimport csv\nimport json\nfrom pathlib import Path\nimport platform\nimport shutil\nimport tempfile\nimport time\nimport uuid\nimport zipfile\n\nimport ocr_pipeline as ocr\n\nMETRICS_VERSION = "bpl-ocr-efficiency-v1"\nSUMMARY_FIELDS = ["model", "run_status", "measured_lines", "device", "hardware", "precision", "setup_s", "load_s",\n                  "processing_s", "mean_s_per_line", "mean_s_per_page", "pages_per_hour",\n                  "peak_ram_mib", "peak_vram_allocated_mib", "gpu_device_peak_mib",\n                  "mean_cpu_pct", "mean_gpu_pct", "export_s", "wall_s"]\n\n\ndef import_efficiency_bundle(archive, root):\n    """Extract the old or measured bundle without regenerating any old results."""\n    root = Path(root)\n    if root.exists() and any(root.iterdir()):\n        raise FileExistsError("Use the already loaded ROOT or a fresh Colab runtime.")\n    with tempfile.TemporaryDirectory() as temporary:\n        staging = Path(temporary) / "data"\n        with zipfile.ZipFile(archive) as bundle:\n            members = [i for i in bundle.infolist() if not i.is_dir() and not any(\n                p.startswith(".") or p == "__MACOSX" for p in Path(i.filename).parts)]\n            indexes = [i.filename for i in members if i.filename.endswith("source/index.json")]\n            if len(indexes) != 1 or sum(i.file_size for i in members) > 500_000_000:\n                raise ValueError("Upload one common source/results ZIP, optionally with efficiency/.")\n            prefix = indexes[0][:-len("source/index.json")]\n            for item in members:\n                if not item.filename.startswith(prefix):\n                    continue\n                relative = Path(item.filename[len(prefix):])\n                if relative.is_absolute() or ".." in relative.parts:\n                    raise ValueError("Unsafe ZIP path")\n                if (not relative.parts or relative.parts[0] not in ("source", "results", "efficiency")\n                        or relative.suffix not in (".jpg", ".json", ".jsonl", ".csv", ".tsv", ".txt")):\n                    continue\n                if item.file_size > 100_000_000:\n                    raise ValueError("ZIP member too large")\n                destination = staging / relative\n                destination.parent.mkdir(parents=True, exist_ok=True)\n                destination.write_bytes(bundle.read(item))\n        index = ocr.load_sources(staging)\n        shutil.copytree(staging, root, dirs_exist_ok=True)\n    return index\n\n\ndef pack_efficiency(root, destination):\n    root, destination = Path(root), Path(destination)\n    if root.resolve() in destination.resolve().parents:\n        raise ValueError("Save transfer ZIP outside the data folder")\n    destination.parent.mkdir(parents=True, exist_ok=True)\n    partial = destination.with_suffix(destination.suffix + ".part")\n    with zipfile.ZipFile(partial, "w", zipfile.ZIP_DEFLATED) as bundle:\n        for name in ("source", "results", "efficiency"):\n            for path in sorted((root / name).rglob("*")):\n                if path.is_file() and not any(p.startswith(".") for p in path.relative_to(root).parts):\n                    bundle.write(path, path.relative_to(root))\n    partial.replace(destination)\n    return destination\n\n\ndef measurement_protocol(index, line_limit=None, warmup_lines=1):\n    all_lines = ocr.visible_lines(index)\n    if line_limit is not None and (type(line_limit) is not int or line_limit < 1):\n        raise ValueError("line_limit must be None or a positive integer")\n    if type(warmup_lines) is not int or warmup_lines < 0:\n        raise ValueError("warmup_lines must be a nonnegative integer")\n    count = min(line_limit or len(all_lines), len(all_lines))\n    positions = [round(i*(len(all_lines)-1)/(count-1)) for i in range(count)] if count > 1 else [0]\n    selected = [all_lines[i] for i in positions]\n    protocol = {"version": METRICS_VERSION, "source_id": index["source_id"],\n                "line_ids": [r["line_id"] for r in selected], "full_scope": count == len(all_lines),\n                "lines_per_page": dict(Counter(str(r["printed_page"]) for r in selected)),\n                "warmup_lines": min(warmup_lines, count), "batch_size": 1,\n                "pipeline_sha256": ocr.sha_file(ocr.__file__), "measurement_sha256": ocr.sha_file(__file__),\n                "scope": index["scope"],\n                "policy": "Fresh line recognition; no prediction cache; original RGB crops; same ordered inputs for every engine."}\n    protocol["protocol_id"] = ocr.identity(protocol)\n    return protocol, selected\n\n\ndef _save_csv(path, rows, fields):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    with path.open("w", encoding="utf-8-sig", newline="") as handle:\n        writer = csv.DictWriter(handle, fieldnames=fields, extrasaction="ignore")\n        writer.writeheader()\n        writer.writerows(rows)\n\n\ndef _mib(value):\n    return value / 1024**2 if value is not None else None\n\n\ndef _reset_before_load(engine, device):\n    """Prevent a previous trial\'s allocator high-water mark leaking into a new run."""\n    if not str(device).startswith(("cuda", "gpu")):\n        return "not_used_cpu"\n    import sys\n    framework = sys.modules.get("paddle" if engine == "paddleocr" else "torch")\n    if framework is None:\n        return "framework_not_loaded; allocator starts with framework initialization"\n    try:\n        cuda = framework.device.cuda if engine == "paddleocr" else framework.cuda\n        cuda.synchronize()\n        names = ("reset_max_memory_allocated", "reset_max_memory_reserved") if engine == "paddleocr" else ("reset_peak_memory_stats",)\n        for name in names:\n            method = getattr(cuda, name, None)\n            if method:\n                method()\n        return "reset_before_load"\n    except Exception as error:\n        return "reset_unavailable: " + type(error).__name__\n\n\ndef _pins(root, engine):\n    path = Path(root) / "results/models.json"\n    models = json.loads(path.read_text()) if path.exists() else {}\n    model = models.get(engine, {})\n    return model.get("settings_history", {}).get(model.get("active_settings_id"), {})\n\n\ndef write_summary(root, protocol_id):\n    """Always retain five rows; never mix samples or present partial runs as full."""\n    root = Path(root)\n    candidates = {engine: [] for engine in ocr.ENGINES}\n    for path in (root / "efficiency/runs").glob("*/*/metrics.json"):\n        try:\n            metric = json.loads(path.read_text())\n            engine = metric.get("engine")\n            if engine in candidates and metric.get("protocol_id") == protocol_id:\n                candidates[engine].append(metric)\n        except (ValueError, OSError):\n            continue\n    rows, selected_runs = [], {}\n    for engine in ocr.ENGINES:\n        matching = candidates[engine]\n        eligible = [m for m in matching if m.get("comparison_eligible")]\n        if not matching:\n            rows.append({"model": engine, "run_status": "not_measured"})\n            continue\n        metric = max(eligible or matching, key=lambda m: m["run_id"])\n        resource = metric.get("resources", {})\n        framework = metric.get("framework_memory", {})\n        metadata = metric.get("metadata", {})\n        actual_device = metric.get("actual_device", "")\n        hardware = metadata.get("gpu_name") or resource.get("gpu_name") if actual_device.startswith(("cuda", "gpu")) else resource.get("cpu_model")\n        selected_runs[engine] = engine + "/" + metric["run_id"]\n        rows.append({"model": engine, "run_status": metric["run_status"],\n                     "measured_lines": metric["measured_lines"], "device": actual_device,\n                     "hardware": hardware, "precision": metric.get("settings", {}).get("precision"),\n                     "setup_s": metric.get("setup_seconds"), "load_s": metric.get("model_load_and_download_seconds"),\n                     "processing_s": metric.get("processing_seconds"),\n                     "mean_s_per_line": metric.get("mean_seconds_per_line"),\n                     "mean_s_per_page": metric.get("mean_seconds_per_page_scope"),\n                     "pages_per_hour": metric.get("page_scopes_per_hour"),\n                     "peak_ram_mib": _mib(resource.get("peak_process_tree_rss_bytes_sampled")),\n                     "peak_vram_allocated_mib": framework.get("gpu_peak_allocated_mib"),\n                     "gpu_device_peak_mib": _mib(resource.get("gpu_peak_used_bytes_sampled")),\n                     "mean_cpu_pct": resource.get("cpu_utilization_percent_one_core"),\n                     "mean_gpu_pct": resource.get("gpu_utilization_percent_mean_sampled"),\n                     "export_s": metric.get("export_seconds"), "wall_s": metric.get("run_wall_seconds")})\n    _save_csv(root / "efficiency/summary.csv", rows, SUMMARY_FIELDS)\n    ocr.write_json(root / "efficiency/summary_scope.json", {\n        "protocol_id": protocol_id, "selected_runs": selected_runs,\n        "selection": "Latest eligible complete run per model, otherwise latest incomplete run. Never combine different samples.",\n        "units": "Seconds; MiB=2^20 bytes; CPU percent: one logical core=100%; GPU utilization/device memory: sampled whole device, not model attribution.",\n        "scope": "Batch1 shared-line recognition. Original page161 scope is body only. No page throughput for smoke/incomplete runs.",\n        "missing_values": "Blank = not measured/unavailable, never zero. GPU memory allocator peaks differ from whole-device sampled memory.",\n        "averages": "CPU/GPU utilization averages cover the whole monitored run (load, warmup, processing, export/cleanup); per-line/page time covers processing only. load_s includes any weight download inside the adapter.",\n        "timing": "wall_s covers load, warmup, processing, local OCR export and cleanup; excludes dependency setup, bundle import, final telemetry serialization and ZIP/browser transfer."})\n    return root / "efficiency/summary.csv"\n\n\ndef run_efficiency(root, engine, *, device="auto", line_limit=None, warmup_lines=1,\n                   setup_seconds=None, import_seconds=None, adapter_factory=None, monitor_factory=None):\n    """Measure an independent fresh run. Existing recognized text is never reused."""\n    root = Path(root)\n    if engine not in ocr.ENGINES:\n        raise ValueError("Unknown engine")\n    # The notebook resolves this during measured setup. Direct callers resolve it\n    # as environment preparation, before the model-run clock starts.\n    if device == "auto" and adapter_factory is None:\n        if engine == "tesseract":\n            device = "cpu"\n        elif engine == "paddleocr":\n            import paddle\n            device = "gpu:0" if paddle.is_compiled_with_cuda() and paddle.device.cuda.device_count() else "cpu"\n        else:\n            import torch\n            device = "cuda:0" if torch.cuda.is_available() else "cpu"\n    index = ocr.load_sources(root)\n    protocol, selected = measurement_protocol(index, line_limit, warmup_lines)\n    protocol_id = protocol["protocol_id"]\n    ocr.write_json(root / "efficiency/protocols" / (protocol_id + ".json"), protocol)\n    run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ") + "_" + uuid.uuid4().hex[:8]\n    directory = root / "efficiency/runs" / engine / run_id\n    directory.mkdir(parents=True)\n    metric = {"version": METRICS_VERSION, "engine": engine, "run_id": run_id,\n              "protocol_id": protocol_id, "source_id": index["source_id"], "requested_lines": len(selected),\n              "measured_lines": 0, "full_scope": protocol["full_scope"], "setup_seconds": setup_seconds,\n              "import_seconds": import_seconds, "run_status": "initializing", "comparison_eligible": False,\n              "batch_size": 1, "warmup_lines": protocol["warmup_lines"], "device_requested": device,\n              "python": platform.python_version(), "platform": platform.platform(),\n              "cache_policy": "Prediction cache ignored. Weights may be cached; combined load/download is measured as observed, not guaranteed cold download.",\n              "timing_scope": "processing includes source image reads, crop, recognizer preprocessing/inference/decode, GPU synchronization, raw logging and progress. No original OCR output is rewritten."}\n    ocr.write_json(directory / "metrics.json", metric)\n    adapter, image, records, page_records, monitor = None, None, [], [], None\n    memory_phases, active_page_info = [], None\n    clock = time.perf_counter()\n    try:\n        monitor = (monitor_factory or ResourceMonitor)(device, engine)\n        monitor.start()\n        started = time.perf_counter()\n        metric["allocator_reset_before_load"] = _reset_before_load(engine, device)\n        metric["model_cache_directory_existed"] = (root.parent / "bpl_efficiency_model_cache").exists()\n        adapter = (adapter_factory or ocr.build_recognizer)(engine, device=device,\n                  pinned_settings=_pins(root, engine), model_cache=root.parent / "bpl_efficiency_model_cache")\n        synchronize(adapter)\n        metric["model_load_and_download_seconds"] = time.perf_counter()-started\n        metric["settings"] = adapter.settings\n        metric["metadata"] = adapter.metadata\n        metric["actual_device"] = getattr(adapter, "device", adapter.metadata.get("actual_device", device))\n        import os\n        metric["thread_environment"] = {k: os.environ.get(k) for k in ("OMP_NUM_THREADS", "OMP_THREAD_LIMIT", "MKL_NUM_THREADS", "OPENBLAS_NUM_THREADS")}\n        framework = getattr(adapter, "framework", None)\n        if framework is not None and hasattr(framework, "get_num_threads"):\n            metric["framework_cpu_threads"] = framework.get_num_threads()\n        memory_phases = [gpu_allocated(adapter)]\n        pages = {p["printed_page"]: p for p in index["pages"]}\n        started = time.perf_counter()\n        for row in selected[:protocol["warmup_lines"]]:\n            from PIL import Image\n            with Image.open(root / pages[row["printed_page"]]["image"]) as original:\n                with original.convert("RGB").crop(tuple(row["bbox"])) as tile:\n                    adapter.predict(tile)\n                    synchronize(adapter)\n        metric["warmup_seconds"] = time.perf_counter()-started\n        memory_phases.append(gpu_allocated(adapter))\n        gpu_allocated(adapter, reset=True)\n        started = time.perf_counter()\n        metric["run_status"] = "processing"\n        try:\n            with (directory / "raw.jsonl").open("w", encoding="utf-8") as log:\n                for number in sorted({r["printed_page"] for r in selected}):\n                    page_clock = time.perf_counter()\n                    page_lines = [r for r in selected if r["printed_page"] == number]\n                    count_before = len(records)\n                    active_page_info = {"printed_page": number, "selected_lines": len(page_lines)}\n                    page = pages[number]\n                    if ocr.sha_file(root / page["image"]) != page["sha256"]:\n                        raise ValueError("Source bytes changed during measurement")\n                    from PIL import Image\n                    with Image.open(root / page["image"]) as original:\n                        image = original.convert("RGB")\n                    for row in page_lines:\n                        synchronize(adapter)\n                        request_clock = time.perf_counter()\n                        record = {"line_id": row["line_id"], "printed_page": number, "bbox": row["bbox"],\n                                  "input_id": ocr.line_identity(index, row)}\n                        try:\n                            with image.crop(tuple(row["bbox"])) as tile:\n                                response = adapter.predict(tile)\n                            synchronize(adapter)\n                            if not isinstance(response.get("raw_text"), str) or response.get("status") not in ("completed", "empty", "truncated"):\n                                raise ValueError("Invalid recognizer response")\n                            record.update(response)\n                        except Exception as error:\n                            oom = "out of memory" in str(error).lower() or "outofmemory" in type(error).__name__.lower()\n                            record.update(raw_text="", status="out_of_memory" if oom else "failed",\n                                          error=type(error).__name__ + ": " + str(error))\n                        record["request_seconds"] = time.perf_counter()-request_clock\n                        record["raw_text_sha256"] = ocr.sha_text(record["raw_text"])\n                        log.write(json.dumps(record, ensure_ascii=False) + "\\n")\n                        log.flush()\n                        records.append(record)\n                        if len(records) % 20 == 0 or len(records) == len(selected):\n                            print(engine, len(records), "/", len(selected), record["status"], flush=True)\n                        if record["status"] == "out_of_memory":\n                            break\n                    image.close()\n                    image = None\n                    page_records.append({**active_page_info, "measured_lines": len(records)-count_before,\n                                         "processing_seconds": time.perf_counter()-page_clock,\n                                         "page_status": "complete" if len(records)-count_before == len(page_lines) else "incomplete"})\n                    active_page_info = None\n                    if records and records[-1]["status"] == "out_of_memory":\n                        break\n        finally:\n            if active_page_info is not None:\n                page_records.append({**active_page_info, "measured_lines": len(records)-count_before,\n                                     "processing_seconds": time.perf_counter()-page_clock, "page_status": "incomplete"})\n            metric["processing_seconds"] = time.perf_counter()-started\n        memory_phases.append(gpu_allocated(adapter))\n        all_attempted = len(records) == len(selected)\n        errors = any(r["status"] in ("failed", "out_of_memory") for r in records)\n        metric["comparison_eligible"] = all_attempted and not errors\n        metric["run_status"] = ("complete" if protocol["full_scope"] else "smoke_complete") if all_attempted and not errors else "incomplete_or_failed"\n    except KeyboardInterrupt:\n        metric["run_status"] = "interrupted"\n        print("Interrupted. Use the download cell to keep the partial measurement.", flush=True)\n    except Exception as error:\n        metric["run_status"] = "failed"\n        metric["error"] = type(error).__name__ + ": " + str(error)\n        print(metric["error"], flush=True)\n    finally:\n        if image is not None:\n            image.close()\n        metric["measured_lines"] = len(records)\n        metric["output_statuses"] = dict(Counter(r["status"] for r in records))\n        metric["request_seconds_sum"] = sum(r["request_seconds"] for r in records)\n        processing = metric.get("processing_seconds")\n        metric["mean_seconds_per_line"] = processing/len(records) if processing and records else None\n        page_count = len({r["printed_page"] for r in selected})\n        full = metric["comparison_eligible"] and protocol["full_scope"]\n        metric["mean_seconds_per_page_scope"] = processing/page_count if full and processing else None\n        metric["page_scopes_per_hour"] = page_count*3600/processing if full and processing else None\n        metric["page_scope_count"] = page_count\n        export_clock = time.perf_counter()\n        page_lookup = {p["printed_page"]: p for p in index["pages"]}\n        _save_csv(directory / "lines.csv", [{"text": " ".join(r["raw_text"].split()),\n                  "sources": page_lookup[r["printed_page"]]["source_page_url"]} for r in records], ["text", "sources"])\n        _save_csv(directory / "pages.csv", page_records,\n                  ["printed_page", "measured_lines", "selected_lines", "processing_seconds", "page_status"])\n        metric["export_seconds"] = time.perf_counter()-export_clock\n        if adapter is not None:\n            try:\n                memory_phases.append(gpu_allocated(adapter))\n            except Exception as error:\n                metric["allocator_error"] = str(error)\n            try:\n                adapter.close()\n            except Exception as error:\n                metric["cleanup_error"] = str(error)\n        metric["framework_memory"] = {"gpu_peak_allocated_mib": max(\n            (_mib(m["peak_allocated_bytes"]) for m in memory_phases if m.get("peak_allocated_bytes") is not None), default=None),\n            "phases": memory_phases, "scope": "Framework allocator; load/warmup/process observed peaks. CPU engines: null."}\n        if monitor is not None:\n            try:\n                metric["resources"] = monitor.stop()\n            except Exception as error:\n                metric["resources"] = {"error": str(error)}\n        metric["run_wall_seconds"] = time.perf_counter()-clock\n        ocr.write_json(directory / "metrics.json", metric)\n        write_summary(root, protocol_id)\n    return metric\n\n\n"""Optional unified resource sampling; no OCR framework is imported here."""\nimport csv\nimport io\nimport platform\nfrom pathlib import Path\nimport shutil\nimport subprocess\nimport threading\nimport time\n\n\ndef _cuda_api(adapter):\n    device = str(getattr(adapter, "device", "cpu"))\n    if not getattr(adapter, "gpu", device.startswith(("cuda", "gpu"))):\n        return None\n    framework = getattr(adapter, "framework", None)\n    if framework is None:\n        raise RuntimeError("GPU adapter exposes no framework for synchronization")\n    return framework.device.cuda if getattr(adapter, "engine", "") == "paddleocr" else framework.cuda\n\n\ndef synchronize(adapter):\n    """Synchronize the selected framework device before/after a timed GPU phase."""\n    cuda = _cuda_api(adapter)\n    if cuda is not None:\n        cuda.synchronize()\n\n\ndef gpu_allocated(adapter, reset=False):\n    """Framework allocator bytes, distinct from whole-device nvidia-smi usage."""\n    cuda = _cuda_api(adapter)\n    result = {"allocated_bytes": None, "peak_allocated_bytes": None,\n              "reserved_bytes": None, "peak_reserved_bytes": None,\n              "scope": "framework allocator; excludes driver and other processes"}\n    if cuda is None:\n        result["status"] = "not_used_cpu"\n        return result\n    errors = []\n    if reset:\n        resetters = (("reset_max_memory_allocated", "reset_max_memory_reserved")\n                     if getattr(adapter, "engine", "") == "paddleocr" else ("reset_peak_memory_stats",))\n        for name in resetters:\n            method = getattr(cuda, name, None)\n            if method:\n                try:\n                    method()\n                except Exception as error:\n                    errors.append(name + ": " + type(error).__name__)\n    for key, name in (("allocated_bytes", "memory_allocated"),\n                      ("peak_allocated_bytes", "max_memory_allocated"),\n                      ("reserved_bytes", "memory_reserved"),\n                      ("peak_reserved_bytes", "max_memory_reserved")):\n        method = getattr(cuda, name, None)\n        if method:\n            try:\n                result[key] = int(method())\n            except Exception as error:\n                errors.append(name + ": " + type(error).__name__)\n    result.update(status="available" if result["allocated_bytes"] is not None else "unavailable", errors=errors)\n    return result\n\n\nclass ResourceMonitor:\n    """Self/reaped-child CPU deltas; process-tree RSS sampled every interval.\n\n    RSS sums can double-count shared pages and miss short-lived child peaks.\n    nvidia-smi values describe the entire selected GPU, not this process alone.\n    """\n    def __init__(self, device, engine, interval=.1, psutil_module=None, subprocess_runner=None):\n        if interval <= 0:\n            raise ValueError("Sampling interval must be positive")\n        self.device, self.engine, self.interval = str(device), engine, float(interval)\n        self._runner = subprocess_runner or subprocess.run\n        self._event, self._lock = threading.Event(), threading.Lock()\n        self._threads, self._result, self._errors = [], None, []\n        self._peak_rss, self._rss_count, self._gpu_samples = None, 0, []\n        self._use_gpu = self.device.startswith(("cuda", "gpu"))\n        self._smi = shutil.which("nvidia-smi") or ("nvidia-smi" if subprocess_runner else None)\n        self._gpu_status = ("not_used_cpu" if not self._use_gpu else\n                            "pending_first_sample" if self._smi else "unavailable: nvidia-smi missing")\n        self._psutil, self._process, self._resource = None, None, None\n        self.hardware = {"cpu_logical_count": None, "cpu_model": platform.processor() or None,\n                         "system_ram_total_bytes": None, "platform": platform.platform()}\n        try:\n            if psutil_module is None:\n                import psutil as psutil_module\n            self._psutil = psutil_module\n            self._process = psutil_module.Process()\n            self.hardware.update(cpu_logical_count=psutil_module.cpu_count(logical=True),\n                                 system_ram_total_bytes=int(psutil_module.virtual_memory().total))\n        except Exception as error:\n            self._error("psutil", error)\n        try:\n            text = Path("/proc/cpuinfo").read_text()\n            self.hardware["cpu_model"] = next((s.split(":", 1)[1].strip() for s in text.splitlines()\n                                               if s.lower().startswith("model name")), self.hardware["cpu_model"])\n        except OSError:\n            pass\n        try:\n            import resource\n            self._resource = resource\n        except ImportError as error:\n            self._error("resource", error)\n\n    def _error(self, scope, error):\n        message = scope + ": " + type(error).__name__ + ": " + str(error)[:140]\n        with self._lock:\n            if message not in self._errors and len(self._errors) < 5:\n                self._errors.append(message)\n\n    def _cpu(self):\n        if self._resource is None:\n            return None\n        try:\n            r = self._resource\n            own, children = r.getrusage(r.RUSAGE_SELF), r.getrusage(r.RUSAGE_CHILDREN)\n            return own.ru_utime + own.ru_stime, children.ru_utime + children.ru_stime\n        except Exception as error:\n            self._error("cpu", error)\n            return None\n\n    def _sample_rss(self):\n        if self._process is None:\n            return\n        try:\n            processes = [self._process] + self._process.children(recursive=True)\n            total = 0\n            for process in processes:\n                try:\n                    total += process.memory_info().rss\n                except (self._psutil.NoSuchProcess, self._psutil.AccessDenied):\n                    continue\n            with self._lock:\n                self._peak_rss = max(self._peak_rss or 0, int(total))\n                self._rss_count += 1\n        except Exception as error:\n            self._error("rss", error)\n\n    def _rss_loop(self):\n        while not self._event.wait(self.interval):\n            self._sample_rss()\n\n    def _gpu_loop(self):\n        command = [self._smi, "--id=0", "--query-gpu=name,memory.used,memory.total,utilization.gpu",\n                   "--format=csv,noheader,nounits"]\n        while not self._event.is_set():\n            try:\n                response = self._runner(command, capture_output=True, text=True, timeout=2, check=True)\n                row = next(csv.reader(io.StringIO(response.stdout)))\n                if len(row) != 4:\n                    raise ValueError("Unexpected GPU query output")\n                def numeric(value):\n                    try:\n                        return float(value.strip())\n                    except ValueError:\n                        return None\n                values = [numeric(value) for value in row[1:]]\n                if not self._event.is_set():\n                    with self._lock:\n                        self._gpu_samples.append((row[0].strip(), *values))\n                        self._gpu_status = "available"\n            except Exception as error:\n                self._gpu_status = "unavailable: " + type(error).__name__\n                self._error("nvidia-smi", error)\n                return\n            if self._event.wait(max(1.0, self.interval)):\n                return\n\n    def start(self):\n        if self._threads or hasattr(self, "_started"):\n            raise RuntimeError("Create a new monitor for each measured run")\n        self._started, self._cpu_start = time.perf_counter(), self._cpu()\n        self._sample_rss()\n        self._threads.append(threading.Thread(target=self._rss_loop, daemon=True))\n        if self._use_gpu and self._smi:\n            self._threads.append(threading.Thread(target=self._gpu_loop, daemon=True))\n        for thread in self._threads:\n            thread.start()\n        return self\n\n    def stop(self):\n        if self._result is not None:\n            return self._result\n        if not hasattr(self, "_started"):\n            raise RuntimeError("Start the monitor before stopping it")\n        self._sample_rss()\n        cpu_end, elapsed = self._cpu(), time.perf_counter() - self._started\n        self._event.set()\n        for thread in self._threads:\n            thread.join(timeout=3)\n        if self._gpu_status == "pending_first_sample":\n            self._gpu_status = "no_sample_before_stop"\n        own = child = None\n        if self._cpu_start is not None and cpu_end is not None:\n            own, child = [max(0.0, end - start) for start, end in zip(self._cpu_start, cpu_end)]\n        used = [r[1] for r in self._gpu_samples if r[1] is not None]\n        total = [r[2] for r in self._gpu_samples if r[2] is not None]\n        util = [r[3] for r in self._gpu_samples if r[3] is not None]\n        self._result = {**self.hardware, "monitor_wall_seconds": elapsed,\n            "cpu_seconds_self": own, "cpu_seconds_reaped_children": child,\n            "cpu_seconds_self_and_reaped_children": None if own is None else own + child,\n            "cpu_utilization_percent_one_core": None if own is None or elapsed <= 0 else (own + child) / elapsed * 100,\n            "peak_process_tree_rss_bytes_sampled": self._peak_rss, "rss_sample_count": self._rss_count,\n            "rss_interval_seconds": self.interval, "gpu_monitor_status": self._gpu_status,\n            "gpu_name": self._gpu_samples[-1][0] if self._gpu_samples else None,\n            "gpu_total_memory_bytes": int(max(total) * 1024**2) if total else None,\n            "gpu_peak_used_bytes_sampled": int(max(used) * 1024**2) if used else None,\n            "gpu_utilization_percent_mean_sampled": sum(util)/len(util) if util else None,\n            "gpu_utilization_percent_peak_sampled": max(util) if util else None,\n            "gpu_sample_count": len(self._gpu_samples), "gpu_interval_seconds": max(1.0, self.interval),\n            "scope": "CPU=self+reaped children (excludes alive/unreaped children); RSS=sum live process tree, sampled, shared pages may double count; GPU=whole device 0, sampled; monitor overhead included",\n            "errors": list(self._errors)}\n        return self._result\n\n\n\ndef build_notebook():\n    """New notebook only; neither existing source nor existing notebook is edited."""\n    import nbformat as nbf\n    directory = Path(__file__).parent\n    existing = json.loads((directory / "ocr_colab.ipynb").read_text())\n    setup_cell = next(c for c in existing["cells"] if c.get("id") == "ocr-setup")\n    setup = "".join(setup_cell["source"])\n    expected = "dependencies = [\'Pillow\', \'numpy\']"\n    if setup.count(expected) != 1:\n        raise ValueError("Existing setup changed; inspect before building the additive notebook")\n    setup = "import time\\n_METRIC_SETUP_START = time.perf_counter()\\n" + setup.replace(\n        expected, "dependencies = [\'Pillow\', \'numpy\', \'psutil\']")\n    setup += "\\nif ENGINE not in (\'tesseract\', \'paddleocr\'):\\n    import torch\\n    DEVICE = \'cuda:0\' if torch.cuda.is_available() else \'cpu\'\\n"\n    setup += "SETUP_SECONDS = time.perf_counter() - _METRIC_SETUP_START\\nprint(\'Measured setup seconds:\', SETUP_SECONDS, \'| actual device:\', DEVICE)\\n"\n    cells = []\n    def md(value):\n        cells.append(nbf.v4.new_markdown_cell(value.strip()))\n    def code(value, name, hidden=False):\n        cell = nbf.v4.new_code_cell(value.strip())\n        cell["id"] = "efficiency-" + name\n        if hidden:\n            cell.metadata.update({"cellView": "form", "collapsed": True, "jupyter": {"source_hidden": True}})\n        cells.append(cell)\n    md("""# BPL OCR · unified efficiency measurement\n\nUse one model per **fresh hosted Colab runtime**. The existing OCR notebook, source scans and\nresults remain unchanged. This notebook runs **fresh inference**, including for models with\nexisting completed predictions. Existing results supply immutable model revisions only.\n\nAll five models receive identical original line crops, batch size 1, with one discarded warmup.\nDefault scope: 1,573 shared detections on pages 161–167; page 161 includes only its directory body.\nTesseract is measured as **line PSM 7**, with an engine process per line; historical page PSM 3\ntimings are a different condition. Device/precision settings follow the existing adapters.\n""")\n    md("""## 1. Choose a model and set up Colab\n\nEdit `ENGINE`, then run both setup/helper cells. Reset the runtime before switching models.\nTesseract/Paddle use CPU; dots/Qwen require GPU; EasyOCR uses available CPU/GPU.\nSetup seconds include dependency installation. Download plus model loading is measured separately.\n""")\n    code(setup, "setup")\n    sources = {"ocr_pipeline.py": (directory / "ocr_pipeline.py").read_text(),\n               "ocr_efficiency.py": Path(__file__).read_text()}\n    code("_METRIC_SOURCES = " + repr(sources) + "\\n"\n         + "_CODE = Path(\'/content/bpl_efficiency_code\')\\n_CODE.mkdir(exist_ok=True)\\n"\n         + "for _name, _source in _METRIC_SOURCES.items():\\n    (_CODE / _name).write_text(_source, encoding=\'utf-8\')\\n"\n         + "sys.path.insert(0, str(_CODE))\\n"\n         + "from ocr_efficiency import import_efficiency_bundle, run_efficiency, pack_efficiency\\n", "helpers", True)\n    md("""## 2. Upload the existing final data ZIP\n\nUpload a ZIP of `fa26/bpl_ocrs/`, or the latest efficiency ZIP from this notebook.\nIt contains `source/`, `results/`, and optionally `efficiency/`. Importing retains existing bytes;\nonly the new efficiency directory receives measurements. Browser upload time is excluded.\n""")\n    code("""from google.colab import files\nuploaded = files.upload()\nnames = [name for name in uploaded if name.lower().endswith(\'.zip\')]\nif len(names) != 1:\n    raise ValueError(\'Upload exactly one common final-results or efficiency ZIP.\')\nimport_started = time.perf_counter()\nuploaded_zip = Path(\'/content/bpl_efficiency_input.zip\')\nuploaded_zip.write_bytes(uploaded[names[0]])\nROOT = Path(\'/content/bpl_efficiency\')\nindex = import_efficiency_bundle(uploaded_zip, ROOT)\nIMPORT_SECONDS = time.perf_counter() - import_started\nprint(\'Seven pages loaded. Existing results retained.\')\n""", "inputs")\n    md("""## 3. Measure fresh recognition\n\nKeep `LINE_LIMIT = None` for the complete comparable measurement. Use `6` only for a smoke test;\nsmoke tests have a distinct protocol and no page-time estimate. Re-running measures again;\nit does not reuse OCR predictions. Warmup is excluded from measured line counts.\n\nSummary metrics include model load, processing, per-line/page averages, throughput, local export,\nrun wall time, CPU usage, sampled process-tree RAM (including Tesseract children), available\nframework GPU allocator peaks, and sampled whole-device GPU memory/utilization.\nUnavailable telemetry stays blank. CPU percent uses one logical core = 100% and may exceed 100%.\nRAM/device samples may miss brief peaks; whole-device GPU readings include other processes.\nCPU/GPU utilization averages cover loading through cleanup; line/page time covers processing only.\n\nProcessing includes page reads, crops, adapter preprocessing/inference/decode, GPU synchronization\nand raw logging. Local export means writing this run\'s OCR CSVs, not ZIP/browser download.\nRun wall time includes load/warmup/processing/export/cleanup, excluding setup, bundle import,\nfinal metric serialization and ZIP transfer. No energy, monetary cost, FLOPs or accuracy is inferred.\n""")\n    code("""LINE_LIMIT = None  # 6 for a smoke test; None for all 1,573 source lines.\nif ENGINE != _SETUP_ENGINE:\n    raise RuntimeError(\'Reset the runtime before changing ENGINE.\')\nmetrics = run_efficiency(ROOT, ENGINE, device=DEVICE, line_limit=LINE_LIMIT,\n                         setup_seconds=SETUP_SECONDS, import_seconds=IMPORT_SECONDS)\nprint(\'Run status:\', metrics[\'run_status\'])\nprint((ROOT / \'efficiency/summary.csv\').read_text(encoding=\'utf-8-sig\'))\n""", "run")\n    md("""## 4. Download and continue with the next model\n\n`efficiency/summary.csv` has one row per model and selects completed comparable measurements.\n`not_measured` means that model still needs this fresh measurement. Each run retains detailed\nsettings, timing, telemetry, per-page records and exact new OCR responses under `efficiency/runs/`.\nThe original `results/lines.csv`, raw outputs and source images stay unchanged.\n\nDownload after completion or interruption. In a fresh runtime, select the next ENGINE and upload\nthis latest ZIP to accumulate all five rows. Use the same LINE_LIMIT for comparable results.\nZIP creation time is printed separately; browser transfer and ZIP time are outside run wall time.\n""")\n    code("""zip_started = time.perf_counter()\narchive = pack_efficiency(ROOT, Path(\'/content/bpl_ocr_efficiency.zip\'))\nprint(\'ZIP creation seconds (outside model run):\', time.perf_counter() - zip_started)\nfiles.download(str(archive))\n""", "export")\n    notebook = nbf.v4.new_notebook(cells=cells, metadata={\n        "kernelspec": {"name": "python3", "display_name": "Python 3", "language": "python"},\n        "language_info": {"name": "python"}, "accelerator": "GPU",\n        "colab": {"name": "ocr_efficiency_colab.ipynb", "gpuType": "T4", "provenance": []}})\n    nbf.validate(notebook)\n    for cell in notebook.cells:\n        if cell.cell_type == "code":\n            compile(cell.source, cell.id, "exec")\n    path = directory / "ocr_efficiency_colab.ipynb"\n    nbf.write(notebook, path)\n    return path\n\n\nif __name__ == "__main__":\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--notebook", action="store_true")\n    parser.add_argument("--pack", type=Path)\n    parser.add_argument("--output", type=Path, default=Path(tempfile.gettempdir()) / "bpl_ocr_efficiency.zip")\n    args = parser.parse_args()\n    if args.notebook:\n        print(build_notebook())\n    elif args.pack:\n        print(pack_efficiency(args.pack, args.output))\n    else:\n        parser.print_help()\n'}
_CODE = Path('/content/bpl_efficiency_code')
_CODE.mkdir(exist_ok=True)
for _name, _source in _METRIC_SOURCES.items():
    (_CODE / _name).write_text(_source, encoding='utf-8')
sys.path.insert(0, str(_CODE))
from ocr_efficiency import import_efficiency_bundle, run_efficiency, pack_efficiency

## 2. Upload the existing final data ZIP

Upload a ZIP of `fa26/bpl_ocrs/`, or the latest efficiency ZIP from this notebook.
It contains `source/`, `results/`, and optionally `efficiency/`. Importing retains existing bytes;
only the new efficiency directory receives measurements. Browser upload time is excluded.

In [ ]:
from google.colab import files
uploaded = files.upload()
names = [name for name in uploaded if name.lower().endswith('.zip')]
if len(names) != 1:
    raise ValueError('Upload exactly one common final-results or efficiency ZIP.')
import_started = time.perf_counter()
uploaded_zip = Path('/content/bpl_efficiency_input.zip')
uploaded_zip.write_bytes(uploaded[names[0]])
ROOT = Path('/content/bpl_efficiency')
index = import_efficiency_bundle(uploaded_zip, ROOT)
IMPORT_SECONDS = time.perf_counter() - import_started
print('Seven pages loaded. Existing results retained.')

## 3. Measure fresh recognition

Keep `LINE_LIMIT = None` for the complete comparable measurement. Use `6` only for a smoke test;
smoke tests have a distinct protocol and no page-time estimate. Re-running measures again;
it does not reuse OCR predictions. Warmup is excluded from measured line counts.

Summary metrics include model load, processing, per-line/page averages, throughput, local export,
run wall time, CPU usage, sampled process-tree RAM (including Tesseract children), available
framework GPU allocator peaks, and sampled whole-device GPU memory/utilization.
Unavailable telemetry stays blank. CPU percent uses one logical core = 100% and may exceed 100%.
RAM/device samples may miss brief peaks; whole-device GPU readings include other processes.
CPU/GPU utilization averages cover loading through cleanup; line/page time covers processing only.

Processing includes page reads, crops, adapter preprocessing/inference/decode, GPU synchronization
and raw logging. Local export means writing this run's OCR CSVs, not ZIP/browser download.
Run wall time includes load/warmup/processing/export/cleanup, excluding setup, bundle import,
final metric serialization and ZIP transfer. No energy, monetary cost, FLOPs or accuracy is inferred.

In [ ]:
LINE_LIMIT = None  # 6 for a smoke test; None for all 1,573 source lines.
if ENGINE != _SETUP_ENGINE:
    raise RuntimeError('Reset the runtime before changing ENGINE.')
metrics = run_efficiency(ROOT, ENGINE, device=DEVICE, line_limit=LINE_LIMIT,
                         setup_seconds=SETUP_SECONDS, import_seconds=IMPORT_SECONDS)
print('Run status:', metrics['run_status'])
print((ROOT / 'efficiency/summary.csv').read_text(encoding='utf-8-sig'))

## 4. Download and continue with the next model

`efficiency/summary.csv` has one row per model and selects completed comparable measurements.
`not_measured` means that model still needs this fresh measurement. Each run retains detailed
settings, timing, telemetry, per-page records and exact new OCR responses under `efficiency/runs/`.
The original `results/lines.csv`, raw outputs and source images stay unchanged.

Download after completion or interruption. In a fresh runtime, select the next ENGINE and upload
this latest ZIP to accumulate all five rows. Use the same LINE_LIMIT for comparable results.
ZIP creation time is printed separately; browser transfer and ZIP time are outside run wall time.

In [ ]:
zip_started = time.perf_counter()
archive = pack_efficiency(ROOT, Path('/content/bpl_ocr_efficiency.zip'))
print('ZIP creation seconds (outside model run):', time.perf_counter() - zip_started)
files.download(str(archive))